# Machine Learning – Unidad 2

## Clasificación binaria de precios de viviendas

**Integrantes:**
- Muñoz Morales, Diego Ignacio
- Villarroel Montecinos, Yenny Vanessa
- Matujara Contreras, Jordan Hernán
- Sepúlveda Alvial, Segundo Alejandro

**Profesor:** Jorge Luis Padilla  
**Dataset:** House Prices – Kaggle  

---


## Introducción

En la Unidad 1 se desarrolló un pipeline reproducible para la predicción del precio de viviendas mediante un modelo de regresión.

En esta Unidad 2, el problema es reformulado como clasificación binaria, con el objetivo de alinearse con las métricas solicitadas en la pauta.

Para ello, se define la variable objetivo HighPrice, donde:
- 1: viviendas con precio sobre la mediana
- 0: viviendas con precio bajo o igual a la mediana

El objetivo es comparar distintos modelos supervisados, evaluar su desempeño y seleccionar el mejor modelo en base a métricas apropiadas.

## Punto 1. Entrenamiento de modelos

Se implementaron distintos modelos de clasificación:

- Regresión logística con penalización (L1, L2 y Elastic Net)
- Random Forest
- XGBoost
- Red neuronal simple

Estos modelos permiten comparar distintos enfoques, desde modelos lineales hasta modelos no lineales más complejos.

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

# Se carga el dataset
df = pd.read_csv("train.csv")

# Procesamiento
if "Id" in df.columns:
    df = df.drop("Id", axis=1)

# Objetivo binario
median_price = df["SalePrice"].median()
df["HighPrice"] = (df["SalePrice"] > median_price).astype(int)

# Imputación
for col in df.select_dtypes(include=["int64", "float64"]).columns:
    if col not in ["SalePrice", "HighPrice"]:
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna("None")

# features
X = df.drop(["SalePrice", "HighPrice"], axis=1)
y = df["HighPrice"]

X = pd.get_dummies(X, drop_first=True)

# split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Función KS
def ks_statistic(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return np.max(tpr - fpr)

# Modelos
from xgboost import XGBClassifier

models = {
    "Logistic_L1": LogisticRegression(penalty="l1", solver="saga", max_iter=5000),
    "Logistic_L2": LogisticRegression(penalty="l2", solver="lbfgs", max_iter=5000),
    "ElasticNet": LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, eval_metric="logloss"),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500)
}

def evaluate(name, model):
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    y_val_prob = model.predict_proba(X_val)[:,1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred),
        "auc": roc_auc_score(y_val, y_val_prob),
        "ks": ks_statistic(y_val, y_val_prob)
    }

results = [evaluate(name, model) for name, model in models.items()]
results_df = pd.DataFrame(results).sort_values(by="auc", ascending=False)
results_df

,model,accuracy,precision,recall,f1,auc,ks
4,XGBoost,0.938356,0.926667,0.952055,0.939189,0.983299,0.883562
3,RandomForest,0.921233,0.942446,0.897260,0.919298,0.980226,0.890411
1,Logistic_L2,0.914384,0.906040,0.924658,0.915254,0.977200,0.856164
0,Logistic_L1,0.869863,0.875000,0.863014,0.868966,0.928504,0.773973
2,ElasticNet,0.869863,0.875000,0.863014,0.868966,0.928504,0.773973
5,NeuralNet,0.743151,0.908046,0.541096,0.678112,0.919708,0.753425


In [5]:
# Reduce complexity (fewer estimators, remove NN for speed)

models_fast = {
    "Logistic_L2": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric="logloss")
}

results_fast = [evaluate(name, model) for name, model in models_fast.items()]
pd.DataFrame(results_fast).sort_values(by="auc", ascending=False)

,model,accuracy,precision,recall,f1,auc,ks
2,XGBoost,0.934932,0.931973,0.938356,0.935154,0.983346,0.883562
1,RandomForest,0.921233,0.936170,0.904110,0.919861,0.978303,0.883562
0,Logistic_L2,0.945205,0.939189,0.952055,0.945578,0.977435,0.890411


## Punto 2. Métricas de desempeño

Se utilizaron las siguientes métricas:

- Accuracy
- Precision
- Recall
- F1-score
- AUC
- KS

Estas métricas permiten evaluar distintos aspectos del desempeño del modelo, especialmente su capacidad de discriminación entre clases.

### Resultados

El modelo XGBoost presentó el mejor desempeño, con un AUC cercano a 0.98.

Esto indica una excelente capacidad para discriminar entre viviendas de alto y bajo precio.

Random Forest también mostró un desempeño competitivo, mientras que la regresión logística obtuvo resultados adecuados pero ligeramente inferiores.

La red neuronal presentó el menor desempeño relativo.

## Punto 3. Optimización de hiperparámetros

Se utilizó RandomizedSearchCV para optimizar el modelo seleccionado.

Este método permite evaluar combinaciones aleatorias de parámetros mediante validación cruzada, reduciendo el costo computacional.

La optimización permitió mejorar el desempeño del modelo, evidenciando la importancia del ajuste de hiperparámetros.

## Punto 4. Selección del modelo final

El modelo seleccionado fue XGBoost, ya que presentó el mayor valor de AUC en la muestra de validación.

Se eligió AUC como métrica principal, debido a que mide la capacidad de discriminación del modelo sin depender de un umbral específico.

Los resultados muestran que el modelo logra una buena generalización, con un equilibrio adecuado entre precisión y recall.

## Punto 5. Repositorio GitHub

El desarrollo fue almacenado en un repositorio GitHub, lo que permite asegurar:

- Reproducibilidad
- Versionado
- Trazabilidad

Repositorio:
https://github.com/diegomunozmo-ops/Trabajo-2-machine-learning

## Conclusión

En este trabajo se reformuló un problema de regresión como clasificación binaria, permitiendo aplicar métricas más adecuadas al contexto solicitado.

Se compararon distintos modelos de machine learning, donde XGBoost destacó como la mejor alternativa.

El uso de métricas como AUC y KS permitió evaluar correctamente el desempeño del modelo, demostrando una alta capacidad de discriminación.

El trabajo evidencia la importancia del preprocesamiento, la selección de modelos y la optimización de hiperparámetros en proyectos de machine learning.

## Bibliografía

- Géron, A. (2022). Hands-On Machine Learning  
- Kaggle – House Prices Dataset  
- Material de clases Unidad 2  